In [1]:
import pandas as pd
import re
import pymorphy2
from pymystem3 import Mystem
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
import random
import transformers
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [2]:
train = pd.read_csv('train.csv') 
test = pd.read_csv('test.csv')
sub_exp = pd.read_csv('submission_example.csv')

In [3]:
cats = pd.read_csv('categories.txt', header=None, names=['cats'])

In [4]:
train

,text
0,"Заказали 14.10.2017 , получили 25.10.2017 \r\n..."
1,"футболка хорошего качества,но футболка не как ..."
2,Все отлично!!!
3,"Рисунок не очень чёткий, а ткань прозрачная, в..."
4,плохо!!!Низ рваный..деньги не вернули!Открыла ...
...,...
1813,"Спасибо,подошло по размеру.Все понравилось в п..."
1814,"доставка быстрая, до Саратова около 2 недель. ..."
1815,"на внешний вид шапка нормальная, на большой об..."
1816,За 4 месяца товар так и не дошел до покупателя.


In [5]:
test

,text
0,Советую продавца
1,По вашему это платье???? Это узкая кофта !!!! ...
2,Жуткая синтетика. Неприятная ткань. Летом не п...
3,"Джемперок так себе на хилую четверку,запах гол..."
4,"обычная х/б рубашка.не плотная,просвечивает ни..."
...,...
7271,"Все отлично, на ОГ 86, ОТ 70, ОБ 99 рост 160 ..."
7272,"так и не пришли, но деньги вернули!"
7273,Рубашка маломерка. XL - по меркам соответствуе...
7274,на картинке совсем другая кофта и модель и ткань


In [6]:
sub_exp

,category
0,бытовая техника
1,нет товара
2,обувь
3,одежда
4,посуда
5,текстиль
6,товары для детей
7,украшения и аксессуары
8,электроника
9,посуда


In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1818 entries, 0 to 1817
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1818 non-null   object
dtypes: object(1)
memory usage: 14.3+ KB


In [8]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7276 entries, 0 to 7275
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    7276 non-null   object
dtypes: object(1)
memory usage: 57.0+ KB


In [9]:
cats

,cats
0,бытовая техника
1,обувь
2,одежда
3,посуда
4,текстиль
5,товары для детей
6,украшения и аксессуары
7,электроника
8,нет товара


In [10]:
train['text'].str.len().idxmax()

13

In [11]:
train['text'][13]

'Качество очень даже неплохое, что хотела, синтетика немного похожа на замшу, Но, размер просто капелюшчный, на 44 размер XXL еле, еле подошёл. Я заказала 3xL, а прислали 2xl, открыла спор, никак не решили проблему. Очень обидно.'

In [13]:
mystem = Mystem()

In [14]:
def clean_lemm_text(text):
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)  
    text = re.sub(r"[^a-zA-Zа-яА-ЯёЁ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    lemmas = mystem.lemmatize(text)
    lemmas = [lemma for lemma in lemmas if lemma.strip()]

    return " ".join(lemmas)

    

In [15]:
train['text'] = train['text'].apply(clean_lemm_text)

In [13]:
model_name = "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

classifier = pipeline("zero-shot-classification", model=model, tokenizer=tokenizer)

text = "Прислал пальто вместо куртки!"
result = classifier(text, candidate_labels=cats['cats'], hypothesis_template="Этот текст относится к {}.")

print(result)

Device set to use mps:0


{'sequence': 'Прислал пальто вместо куртки!', 'labels': ['одежда', 'посуда', 'обувь', 'украшения и аксессуары', 'нет товара', 'текстиль', 'электроника', 'товары для детей', 'бытовая техника'], 'scores': [0.32553425431251526, 0.3056989908218384, 0.11597806960344315, 0.10674961656332016, 0.042526960372924805, 0.031499043107032776, 0.02541753463447094, 0.024178702384233475, 0.0224168561398983]}


In [19]:
def classifier_cats(text):
    result = classifier(text, candidate_labels=cats['cats'], hypothesis_template = "Этот текст относится к {}.")
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    
    final_label = top_label if top_score >= 0.3 else "нет товара"
    
    return top_label, final_label

In [20]:
train[['auto_cats', 'auto_cats_bor']] = train['text'].apply(classifier_cats).apply(pd.Series)

In [22]:
train.to_csv(
    'train_with_cats.csv',
    index= False,
    encoding = 'utf-8'
)

In [18]:
train_cats = pd.read_csv('train_with_cats.csv')[['text', 'auto_cats_bor']] 

In [19]:
train_cats

,text,auto_cats_bor
0,заказывать 14 10 2017 получать 25 10 2017 на м...,одежда
1,футболка хороший качество но футболка не как д...,одежда
2,все отлично,украшения и аксессуары
3,рисунок не очень четкий а ткань прозрачный вид...,текстиль
4,плохо низ рваный деньги не вернуть открывать с...,нет товара
...,...,...
1813,спасибо подходить по размер все понравиться в ...,одежда
1814,доставка быстрый до саратов около 2 неделя упа...,одежда
1815,на внешний вид шапка нормальный на большой объ...,одежда
1816,за 4 месяц товар так и не доходить до покупатель,одежда


In [14]:
paraphraser_model = "cointegrated/rut5-base-paraphraser"
parap_model = T5ForConditionalGeneration.from_pretrained(paraphraser_model)
parap_tokenizer = T5Tokenizer.from_pretrained(paraphraser_model)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
parap_model.to(device)
parap_model.eval()

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


T5ForConditionalGeneration(
  (shared): Embedding(30000, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(30000, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [15]:
def paraphrase_batch(texts, model, tokenizer, num_return_sequences=2, max_length=64):
    inputs = tokenizer(
        [f"Переформулируй это предложение: {t}" for t in texts],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.9
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    result = []
    for i in range(len(texts)):
        result.extend(decoded[i*num_return_sequences:(i+1)*num_return_sequences])
    return result

In [16]:
augmented_df = pd.DataFrame()

In [ ]:
target_sizes = {
    "одежда": None,          
    "электроника": 1000,
    "бытовая техника": 1000
}

default_target_size = 2500

for cat in tqdm(train_cats["auto_cats_bor"].unique(), desc="Categories"):
    subset = train_cats[train_cats["auto_cats_bor"] == cat]
    current_size = len(subset)

    if cat in target_sizes:
        target_size = target_sizes[cat]
        if target_size is None:
            continue 
    else:
        target_size = default_target_size

    needed = max(0, target_size - current_size)
    if needed == 0:
        continue

    augmented_rows = []

    while len(augmented_rows) < needed:
        rows = subset.sample(min(16, needed - len(augmented_rows)), replace=True)
        paraphrased_texts = paraphrase_batch(
            rows["text"].tolist(),
            parap_model,
            parap_tokenizer,
            num_return_sequences=2
        )

        for t, (_, row) in zip(paraphrased_texts, rows.iterrows()):
            augmented_rows.append({"text": t, "auto_cats_bor": cat})
            if len(augmented_rows) >= needed:
                break

    augmented_rows = augmented_rows[:needed]

    augmented_df = pd.concat([augmented_df, pd.DataFrame(augmented_rows)], ignore_index=True)

Categories:  56%|█████████████████▊              | 5/9 [36:16<32:01, 480.32s/it]

In [23]:
train_cats = pd.concat([train_cats, augmented_df], ignore_index=True)

In [24]:
train_cats['auto_cats_bor'].value_counts()

auto_cats_bor
одежда                    1198
украшения и аксессуары    1000
текстиль                  1000
нет товара                1000
посуда                    1000
обувь                     1000
товары для детей          1000
электроника               1000
бытовая техника           1000
Name: count, dtype: int64

In [25]:
train_cats = train_cats.sample(frac=1, random_state=42).reset_index(drop=True)

In [31]:
train_cats

,text,auto_cats_bor
0,Переформулируйте предложение: пойти по 35 суто...,текстиль
1,Переформулируйте это предложение: солнечный пр...,бытовая техника
2,Переформулируйте это предложение: stunning оче...,бытовая техника
3,Переформулируйте это предложение,посуда
4,Приготовь это предложение: задержать 60 минут ...,нет товара
...,...,...
9193,Укажите это предложение из этого слова: это ху...,обувь
9194,Переформулируйте это предложение.,посуда
9195,Переформулируйте это предложение,обувь
9196,синтетика в центр грудь проходить шов не ровны...,нет товара


In [27]:
train_cats.to_csv("train_aug_labeled.csv", index=False, encoding="utf-8")

In [42]:
balanced_train['auto_cats_bor'].value_counts()

auto_cats_bor
одежда                    1198
украшения и аксессуары    1000
текстиль                  1000
нет товара                1000
посуда                    1000
обувь                     1000
товары для детей          1000
электроника               1000
бытовая техника           1000
Name: count, dtype: int64

In [61]:
train = pd.read_csv('labeled_data.csv')

In [46]:
balanced_train.to_csv("balanced_data_cats.csv", index=False, encoding="utf-8")

In [15]:
balanced_train = pd.read_csv("balanced_data_cats.csv", encoding="utf-8")

In [33]:
device = torch.device("cpu")
model.to(device)

XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 384, padding_idx=1)
      (position_embeddings): Embedding(514, 384, padding_idx=1)
      (token_type_embeddings): Embedding(1, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=384, out_features=38

In [34]:
train_df, valid_df = train_test_split(
    train_cats,
    test_size=0.2,
    stratify=train_cats["auto_cats_bor"],
    random_state=42
)

In [35]:
le = LabelEncoder()
y_train = le.fit_transform(train_df["auto_cats_bor"])
y_valid = le.transform(valid_df["auto_cats_bor"])

In [36]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(le.classes_),
    ignore_mismatched_sizes=True
)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([9]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([3, 384]) in the checkpoint and torch.Size([9, 384]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [37]:
lora_config = LoraConfig(
    r=8,           
    lora_alpha=16,        
    target_modules=["query", "key", "value"],  
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config)

In [38]:
def apply_prompt(text):
    return f"К какой категории относится следующий комментарий? {text}"

In [39]:
train_texts = [apply_prompt(t) for t in train_df["text"].tolist()]
valid_texts = [apply_prompt(t) for t in valid_df["text"].tolist()]

In [40]:
train_encodings = tokenizer(
    train_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

valid_encodings = tokenizer(
    valid_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [41]:
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TextDataset(train_encodings, y_train)
valid_dataset = TextDataset(valid_encodings, y_valid)


In [42]:

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=4e-4,
    logging_steps=50,
    save_strategy="epoch",
    fp16=False, 
    save_total_limit=1
)

In [43]:
def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    labels = p.label_ids
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "weighted_f1": f1}

In [44]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)

In [45]:
trainer.train()

print("Evaluating on validation dataset...")
metrics = trainer.evaluate(eval_dataset=valid_dataset)
print(metrics)

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,2.159300
100,1.924300
150,1.771100
200,1.706300
250,1.600200
300,1.622400
350,1.556300
400,1.634000
450,1.575000
500,1.541000


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Evaluating on validation dataset...


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 1.141845703125, 'eval_accuracy': 0.6, 'eval_weighted_f1': 0.5824177220188015, 'eval_runtime': 3.6561, 'eval_samples_per_second': 503.263, 'eval_steps_per_second': 31.454, 'epoch': 5.0}


In [46]:
texts = [f"К какой категории относится следующий комментарий? {t}" for t in valid_df["text"].tolist()]
all_preds = []

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
model.eval()

with torch.no_grad():
    for i in tqdm(range(0, len(texts), 8), desc="Inference"):
        batch_texts = texts[i:i+8]
        encodings = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
        encodings = {k: v.to(device) for k, v in encodings.items()}
        outputs = model(**encodings)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.tolist())

pred_labels = le.inverse_transform(all_preds)
valid_df["pred_category"] = pred_labels

Inference: 100%|██████████████████████████████| 230/230 [00:11<00:00, 20.85it/s]


In [47]:
y_true = valid_df["auto_cats_bor"]          
y_pred = pred_labels                       

f1_weighted = f1_score(y_true, y_pred, average="weighted")
print(f"Weighted F1: {f1_weighted:.4f}")

Weighted F1: 0.5824


In [48]:
print(classification_report(y_true, y_pred, digits=4))

                        precision    recall  f1-score   support

       бытовая техника     0.9151    0.9700    0.9417       200
            нет товара     0.4564    0.3400    0.3897       200
                 обувь     0.3081    0.3050    0.3065       200
                одежда     0.7429    0.9750    0.8432       240
                посуда     0.3136    0.1850    0.2327       200
              текстиль     0.4047    0.4350    0.4193       200
      товары для детей     0.5949    0.7050    0.6453       200
украшения и аксессуары     0.6290    0.5850    0.6062       200
           электроника     0.7857    0.8250    0.8049       200

              accuracy                         0.6000      1840
             macro avg     0.5723    0.5917    0.5766      1840
          weighted avg     0.5760    0.6000    0.5824      1840

